# Neural Network Fundamentals



---

## 1. Learning Objectives

- Describe biological inspiration for artificial neural networks.
- Compute a single neuron's weighted sum and apply activation functions.
- Explain weights, bias, layers (input, hidden, output).
- Implement a two-layer neural network forward pass using NumPy.
- Train an MLPClassifier and compare different activation functions.
- Build a 3-layer XOR solver from scratch.

## Concept / Theory

### Biological Inspiration
An artificial neuron mimics a biological neuron:
- **Dendrites** → Input features
- **Cell body** → Weighted sum + activation
- **Axon** → Output signal

### Neuron Math
```
z = x₁w₁ + x₂w₂ + ... + xₙwₙ + b
output = activation(z)
```

### Key Terms
| Term | Meaning |
|---|---|
| **Weight (w)** | Learned importance of each input |
| **Bias (b)** | Shifts activation threshold |
| **Activation** | Introduces non-linearity |
| **Layer** | Group of neurons |

### Network Architecture
```
Input Layer  → Hidden Layers → Output Layer
[x₁,x₂,...xₙ] → [neurons]   → [prediction]
```

### Backpropagation (Concept)
The learning algorithm that computes how much each weight contributes to the error, then adjusts weights in the direction that reduces error (gradient descent).

In [ ]:
# Setup — NumPy only for theory labs
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import load_digits, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
print('Libraries ready!')

## Practical Lab 1 – Single Artificial Neuron

### Objective
Compute the weighted sum of a single neuron manually and with NumPy.

### Explanation
Every neuron computes: **z = w·x + b**, then applies an activation function. Understanding this is the foundation of all deep learning.

In [ ]:
# Lab 1: Single Neuron Calculation
# Example: predict if a student passes based on 2 features
# Inputs: study hours (x1=7), attendance% (x2=80)
x1, x2 = 7.0, 80.0

# Learned weights and bias
w1, w2 = 0.5, 0.02
b       = -5.0

# Weighted sum (pre-activation)
z = x1*w1 + x2*w2 + b
print(f'Inputs: x1={x1}, x2={x2}')
print(f'Weights: w1={w1}, w2={w2}, bias={b}')
print(f'Weighted sum z = {x1}*{w1} + {x2}*{w2} + ({b}) = {z:.2f}')

In [ ]:
# Apply sigmoid activation → probability
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

output = sigmoid(z)
print(f'Sigmoid output (probability of passing): {output:.4f}')
print(f'Prediction: {"Pass" if output >= 0.5 else "Fail"}')

# NumPy vector version (multiple inputs at once)
X = np.array([[7.0, 80], [3.0, 50], [8.0, 90], [2.0, 40]])
W = np.array([w1, w2])
Z = X @ W + b       # matrix multiplication
Y = sigmoid(Z)
for i, (xi, yi) in enumerate(zip(X, Y)):
    print(f'  Student {i+1}: hours={xi[0]}, att={xi[1]:.0f}% → prob={yi:.3f} → {"Pass" if yi>=0.5 else "Fail"}')

### Trainee Questions
1. What would happen if we increase w1 (study hours weight)?
2. What is the role of the bias term?
3. Why do we apply sigmoid to z instead of using z directly?

## Practical Lab 2 – Activation Functions

### Objective
Implement and visualize Step, Sigmoid, Tanh, and ReLU functions.

### Explanation
Activation functions introduce **non-linearity** — without them, stacking layers would just produce another linear model. **ReLU** is the default in most modern deep networks.

In [ ]:
# Lab 2: Activation Functions
def step(z):    return np.where(z >= 0, 1, 0)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def tanh(z):    return np.tanh(z)
def relu(z):    return np.maximum(0, z)
def leaky_relu(z, alpha=0.01): return np.where(z >= 0, z, alpha*z)

z = np.linspace(-5, 5, 300)
functions = [('Step',     step,       'darkgreen'),
             ('Sigmoid',  sigmoid,    'blue'),
             ('Tanh',     tanh,       'orange'),
             ('ReLU',     relu,       'red'),
             ('Leaky ReLU', leaky_relu, 'purple')]

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (name, fn, color) in zip(axes, functions):
    ax.plot(z, fn(z), color=color, lw=2.5)
    ax.axhline(0, color='gray', lw=0.8); ax.axvline(0, color='gray', lw=0.8)
    ax.set_title(name, fontsize=11); ax.set_xlabel('z'); ax.grid(alpha=0.3)
plt.suptitle('Activation Functions', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Properties comparison
z_test = np.array([-3, -1, 0, 1, 3])
print(f'{"z":>5} | {"Step":>8} | {"Sigmoid":>8} | {"Tanh":>8} | {"ReLU":>8}')
print('-' * 50)
for zv in z_test:
    print(f'{zv:>5.0f} | {step(zv):>8.4f} | {sigmoid(zv):>8.4f} | {tanh(zv):>8.4f} | {relu(zv):>8.4f}')

print('\nActivation Function Use Cases:')
print('  Step     → Simple binary threshold (rarely used now)')
print('  Sigmoid  → Output layer for binary classification')
print('  Tanh     → RNNs, when output should be in [-1,1]')
print('  ReLU     → Default for hidden layers (fast, effective)')
print('  Leaky ReLU → Avoids dead neurons problem of ReLU')

## Practical Lab 3 – 2-Layer Network Forward Pass (NumPy)

### Objective
Implement a complete forward pass through a 2-layer neural network.

### Explanation
A forward pass computes the output layer-by-layer: **H = activation(XW₁ + b₁)** then **Y = activation(HW₂ + b₂)**.

In [ ]:
# Lab 3: 2-Layer Neural Network from Scratch
np.random.seed(42)

# Architecture: 2 inputs → 4 hidden → 1 output
n_input  = 2
n_hidden = 4
n_output = 1

# Initialize weights (Xavier init)
W1 = np.random.randn(n_input, n_hidden)  * np.sqrt(1/n_input)
b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, n_output) * np.sqrt(1/n_hidden)
b2 = np.zeros((1, n_output))

print('Network architecture:')
print(f'  Input:  {n_input} neurons')
print(f'  Hidden: {n_hidden} neurons (ReLU)')
print(f'  Output: {n_output} neuron (Sigmoid)')
print(f'\nW1 shape: {W1.shape} | W2 shape: {W2.shape}')

In [ ]:
# Forward pass function
def relu(z): return np.maximum(0, z)
def sigmoid(z): return 1/(1+np.exp(-z))

def forward_pass(X, W1, b1, W2, b2):
    # Hidden layer
    Z1 = X @ W1 + b1      # pre-activation
    A1 = relu(Z1)         # activation
    # Output layer
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)      # output probability
    return A1, A2

# Test with 4 sample inputs (study_hours, attendance)
X_sample = np.array([[7.0, 80],
                      [3.0, 50],
                      [8.5, 92],
                      [2.0, 45]])

A1, output = forward_pass(X_sample, W1, b1, W2, b2)
print('Hidden activations (A1):')
print(A1.round(3))
print('\nOutput probabilities:')
for i, p in enumerate(output.flatten()):
    print(f'  Sample {i+1}: {p:.4f} → {"Pass" if p>=0.5 else "Fail"}')

### Trainee Questions
1. What is the shape of Z1? Why?
2. Why does ReLU produce sparse (many-zero) hidden activations?
3. What happens if all weights are initialized to 0?

## Practical Lab 4 – MLPClassifier on Digits Dataset

### Objective
Train a multi-layer perceptron using Scikit-learn's MLPClassifier.

### Explanation
MLPClassifier abstracts away the manual weight initialization and backpropagation. We can experiment with architectures easily.

In [ ]:
# Lab 4: MLPClassifier on Digits
digits = load_digits()
X, y   = digits.data, digits.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# Train MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    max_iter=200,
    random_state=42,
    verbose=False
)
mlp.fit(X_tr, y_train)
y_pred = mlp.predict(X_te)

print(f'MLP Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Architecture: input({X.shape[1]}) → 128 → 64 → output({len(np.unique(y))})')

In [ ]:
# Visualize training loss and wrong predictions
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Training loss curve
axes[0].plot(mlp.loss_curve_, color='blue', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('MLP Training Loss Curve')
axes[0].grid(alpha=0.3)

# Show misclassified samples
y_pred_all = mlp.predict(X_tr)  # train predictions
wrong = np.where(y_train != y_pred_all)[0][:8]
for i, ax in enumerate(axes[1:1]):
    pass  # placeholder

# Show some test samples
plt.figure(figsize=(12, 2.5))
for i in range(10):
    plt.subplot(1, 10, i+1)
    plt.imshow(digits.images[X_test.shape[0]//10*i], cmap='gray_r')
    pred = y_pred[X_test.shape[0]//10*i]
    true = y_test[X_test.shape[0]//10*i]
    color = 'green' if pred==true else 'red'
    plt.title(f'P:{pred}\nT:{true}', color=color, fontsize=8)
    plt.axis('off')
plt.suptitle('MLP Predictions (green=correct, red=wrong)')
plt.tight_layout(); plt.show()
plt.figure(figsize=(7,4))
plt.plot(mlp.loss_curve_)
plt.title('Loss Curve'); plt.xlabel('Iteration'); plt.show()

## Practical Lab 5 – Comparing Activations & Learning Rates

### Objective
Experiment with different activation functions and learning rates.

### Explanation
Activation choice and learning rate significantly impact convergence speed and final accuracy.

In [ ]:
# Lab 5: Systematic comparison
results = []
for activation in ['relu', 'tanh', 'logistic']:
    for lr in [0.001, 0.01]:
        mlp_exp = MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation=activation,
            learning_rate_init=lr,
            max_iter=100,
            random_state=42
        )
        mlp_exp.fit(X_tr, y_train)
        acc = accuracy_score(y_test, mlp_exp.predict(X_te))
        results.append({'Activation': activation, 'LR': lr, 'Accuracy': acc})

import pandas as pd
df_res = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)
print('Activation × Learning Rate Comparison:')
print(df_res.to_string(index=False))
print('\n→ ReLU with lr=0.001 is typically best for this task.')

## Exercises

**Exercise 1:** Manually compute the output of a neuron with x=[2,3,1], w=[0.5,-0.2,0.8], b=0.1. Apply ReLU.

In [ ]:
# Exercise 1 — Manual neuron computation
x = np.array([2, 3, 1])
w = np.array([0.5, -0.2, 0.8])
b = 0.1
# Compute z and apply ReLU

**Exercise 2:** Draw (in markdown) the architecture of a 4-layer network: Input(3) → Hidden1(8) → Hidden2(4) → Output(1).

Exercise 2 answer:
```
[x1, x2, x3]  →  [8 neurons, ReLU]  →  [4 neurons, ReLU]  →  [1 neuron, Sigmoid]
 (Input layer)    (Hidden layer 1)      (Hidden layer 2)      (Output)
```

**Exercise 3:** Train MLPClassifier on Iris dataset with hidden layers (32,16) and (64,32,16). Compare accuracy.

In [ ]:
# Exercise 3
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
iris = load_iris()
X_i, y_i = iris.data, iris.target
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(X_i, y_i, test_size=0.2, stratify=y_i, random_state=42)
sc = StandardScaler()
# Compare (32,16) vs (64,32,16)

## Challenge Task – XOR Problem

The XOR problem cannot be solved by a single-layer network (it's not linearly separable). Build a 2-layer network from scratch (no MLPClassifier) that solves XOR using manual forward pass and gradient descent.

In [ ]:
# Challenge Task: XOR from scratch
np.random.seed(42)

# XOR data
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([[0],[1],[1],[0]])  # XOR outputs

# Initialize 2-layer network: 2 → 4 → 1
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros((1, 1))

def sigmoid(z): return 1/(1+np.exp(-z))
def sigmoid_d(a): return a * (1 - a)

lr = 0.5
loss_history = []

for epoch in range(5000):
    # Forward
    Z1 = X_xor @ W1 + b1; A1 = sigmoid(Z1)
    Z2 = A1  @ W2 + b2;  A2 = sigmoid(Z2)
    
    # Loss (binary cross-entropy)
    loss = -np.mean(y_xor*np.log(A2+1e-9) + (1-y_xor)*np.log(1-A2+1e-9))
    loss_history.append(loss)
    
    # Backward
    dA2 = A2 - y_xor
    dW2 = A1.T @ dA2 / 4;  db2 = dA2.mean(axis=0)
    dA1 = dA2 @ W2.T * sigmoid_d(A1)
    dW1 = X_xor.T @ dA1 / 4; db1 = dA1.mean(axis=0)
    
    # Update
    W2 -= lr*dW2; b2 -= lr*db2
    W1 -= lr*dW1; b1 -= lr*db1

print('XOR Predictions after training:')
for xi, yi, pi in zip(X_xor, y_xor, A2):
    print(f'  {xi} → target={yi[0]} | pred={pi[0]:.4f} → {int(pi[0]>=0.5)}')

plt.figure(figsize=(7,4))
plt.plot(loss_history, color='red'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('XOR Network Training Loss'); plt.grid(alpha=0.3); plt.show()

## Real-World AI/ML Application

| Concept | Real Application |
|---|---|
| **Neuron (sigmoid)** | Logistic unit in older networks, output layer |
| **ReLU activation** | Default in CNNs, ResNet, BERT |
| **Backpropagation** | Training every modern deep learning model |
| **Weights & biases** | Learned during training on GPUs |
| **XOR problem** | Historical milestone — proved need for multi-layer networks |

## Day Summary

| Concept | Key Point |
|---|---|
| Neuron | z = Σ(xᵢwᵢ) + b → activation(z) |
| Weights | Learned parameters controlling input importance |
| Bias | Shifts activation threshold independently |
| Activation | Non-linear transform (ReLU, Sigmoid, Tanh) |
| Forward pass | Layer-by-layer computation |
| Backpropagation | Gradient-based weight update |
| MLPClassifier | Scikit-learn's multi-layer perceptron |

**→ Day 8: Deep Learning with TensorFlow/Keras**